<a href="https://colab.research.google.com/github/nguyenhoangnguyenazit-commits/ISYS2001-A2/blob/main/isys2012_a2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Budget Breaker
### ISYS2001 — Assessment 2: Programming Project

A finance assistant that helps university students in Perth track spending, understand where their money goes and check whether they're hitting their savings goal yet. Inspired by the "Spending break down" feature in the CommBank app.


## 1. Understand the Problem

**Business Problem:** As a university student living in Perth, it is easy to lose track of minor daily expenses like coffee, transport, and eating out. By the end of the month I often don't know where my money went and whether I hit my savings target.

**Solution:** A "Budget Breaker" finance assistant that automatically processes monthly transaction data, categorises expenses, calculates what percentage of income is spent per category, and reports whether the monthly savings goal was achieved. A Gemini-powered chat assistant then gives friendly, grounded advice based on those real numbers.

**Target User:** University students who need a quick, no-fuss way to track their living expenses and savings without manually calculating every receipt.

## 2. Identify the Inputs and Outputs

**Inputs:**
- `csv_source` (file path or file-like object): monthly transactions with columns `Date, Amount, Category, Description`
- `monthly_income` (float): total expected income for the month
- `saving_goal` (float): target amount to save by the end of the month

**Outputs:** a dictionary containing:
- `category_totals` — total spent per category
- `category_percentages` — % of income spent per category
- `total_spent` — sum of all expenses
- `remaining_balance` — `monthly_income - total_spent`
- `goal_achieved` — `True` if `remaining_balance >= saving_goal`


##3. Work an Example by Hand

Take a small sample of 4 transactions and work the maths manually, before writing any code.

| Date | Amount | Category |
|---|---|---|
| 2023-10-01 | 18.50 | Rent |
| 2023-10-01 | 5.20 | Coffee |
| 2023-10-02 | 42.30 | Groceries |
| 2023-10-09 | 6.10 | Coffee |

Assume `monthly_income = 1500`, `saving_goal = 300`.

**By hand:**
- Category totals: `Rent = 18.50`, `Coffee = 5.20 + 6.10 = 11.30`, `Groceries = 42.30`
- `total_spent = 18.50 + 11.30 + 42.30 = 72.10`
- `remaining_balance = 1500 - 72.10 = 1427.90`
- `goal_achieved = 1427.90 >= 300` → **True**
- Percentages: `Rent = 18.50/1500*100 = 1.23%`, `Coffee = 11.30/1500*100 = 0.75%`, `Groceries = 42.30/1500*100 = 2.82%`

This worked example is what Test 1 in the testing section checks against later, using a slightly larger sample.

##4. Pseudocode

```
FUNCTION break_down_budget(csv_source, monthly_income, saving_goal):
    TRY:
        READ csv_source into a table of rows (Date, Amount, Category, Description)
        IF required columns are missing THEN RAISE an error
        IF the table has no rows THEN RAISE an error
        CONVERT the Amount column to numbers
        IF any Amount fails to convert THEN RAISE an error
        IF monthly_income <= 0 THEN RAISE an error
        IF saving_goal < 0 THEN RAISE an error

        FOR each unique category in the table:
            category_totals[category] = SUM of Amount where Category = category

        total_spent = SUM of all values in category_totals
        remaining_balance = monthly_income - total_spent

        FOR each category, total in category_totals:
            category_percentages[category] = (total / monthly_income) * 100

        goal_achieved = (remaining_balance >= saving_goal)

        RETURN a dictionary with category_totals, category_percentages,
               total_spent, remaining_balance, goal_achieved

    CATCH any error:
        PRINT a clear error message
        RETURN None
```

## Step 5: Convert to Python

The pseudocode above becomes the `break_down_budget()` function below.


In [5]:
def break_down_budget(csv_source, monthly_income, saving_goal):
    """
    Analyse a user's monthly transactions against their income and savings goal.
    """
    try:
        # --- Load the data ---
        df = pd.read_csv(csv_source)

        # --- Validate required columns exist ---
        required_columns = {"Amount", "Category"}
        missing = required_columns - set(df.columns)
        if missing:
            raise ValueError(f"CSV is missing required columns: {missing}")

        # --- Validate there is actually data ---
        if df.empty:
            raise ValueError("CSV file has no transaction rows.")

        # --- Make sure Amount is numeric (protects against dirty data) ---
        df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
        if df["Amount"].isna().any():
            raise ValueError("Some 'Amount' values are not valid numbers.")

        # --- Validate income/goal are sane numbers ---
        if monthly_income <= 0:
            raise ValueError("monthly_income must be greater than 0.")
        if saving_goal < 0:
            raise ValueError("saving_goal cannot be negative.")

        # --- Core analysis ---
        category_totals = df.groupby("Category")["Amount"].sum().to_dict()
        total_spent = sum(category_totals.values())
        remaining_balance = monthly_income - total_spent
        category_percentages = {
            cat: (amount / monthly_income) * 100
            for cat, amount in category_totals.items()
        }
        goal_achieved = remaining_balance >= saving_goal

        return {
            "category_totals": category_totals,
            "category_percentages": category_percentages,
            "total_spent": total_spent,
            "remaining_balance": remaining_balance,
            "goal_achieved": goal_achieved
        }

    except FileNotFoundError:
        print("Error processing budget: file not found.")
        return None
    except ValueError as e:
        print(f"Error processing budget: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error processing budget: {e}")
        return None

In [4]:
import io

# Mock CSV data (bigger sample than the worked example)
csv_data = """Date,Amount,Category,Description
2026-10-01,18.50,Rent,Weekly share house rent portion
2026-10-01,5.20,Coffee,Flat white at uni cafe
2026-10-02,42.30,Groceries,Woolworths weekly shop
2026-10-02,4.80,Transport,SmartRider top up
2026-10-03,15.00,Eating Out,Lunch with classmates
"""

# Test 1: Happy path — check specific expected values
result = break_down_budget(io.StringIO(csv_data), 1500, 300)
assert result["total_spent"] == 85.80
assert round(result["remaining_balance"], 2) == 1414.20
assert result["goal_achieved"] == True
assert result["category_totals"]["Coffee"] == 5.20

# Test 2: Goal not achieved
result_fail = break_down_budget(io.StringIO(csv_data), 1500, 1450)
assert result_fail["goal_achieved"] == False

# Test 3: File doesn't exist
result_error = break_down_budget("non_existent_file.csv", 1000, 500)
assert result_error is None

# Test 4: Missing required column (edge case)
bad_csv = "Date,Amount\n2026-10-01,50\n"
result_missing_col = break_down_budget(io.StringIO(bad_csv), 1000, 500)
assert result_missing_col is None

# Test 5: Empty file (edge case)
empty_csv = "Date,Amount,Category\n"
result_empty = break_down_budget(io.StringIO(empty_csv), 1000, 500)
assert result_empty is None

# Test 6: Invalid income (edge case)
result_bad_income = break_down_budget(io.StringIO(csv_data), -500, 500)
assert result_bad_income is None

# Test 7: Non-numeric amount (edge case)
dirty_csv = "Date,Amount,Category\n2026-10-01,fifty,Groceries\n"
result_dirty = break_down_budget(io.StringIO(dirty_csv), 1000, 500)
assert result_dirty is None

print("All tests passed successfully!")

NameError: name 'io' is not defined

In [6]:
import getpass

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent?key={GEMINI_API_KEY}"

FINANCE_PERSONA = """You are Budget Breaker, a friendly and encouraging personal finance coach for
university students in Australia. Keep your tone casual, warm, and practical - like a smart friend,
not a formal financial advisor. Give short, specific, and actionable advice.
If the user asks about something unrelated to money, budgeting, or saving, politely say you're a
finance-only assistant and steer the conversation back to their budget."""


def ask_finance_assistant(user_message, budget_summary=None):
    context = ""
    if budget_summary:
        context = f"\n\nHere is the user's current budget summary: {budget_summary}"

    payload = {
        "system_instruction": {"parts": [{"text": FINANCE_PERSONA}]},
        "contents": [{"parts": [{"text": user_message + context}]}]
    }

    try:
        response = requests.post(GEMINI_URL, json=payload, timeout=15)
        response.raise_for_status()
        data = response.json()
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except requests.exceptions.RequestException as e:
        return f"Sorry, I couldn't reach the assistant right now ({e})."
    except (KeyError, IndexError):
        return "Sorry, I got a response I couldn't understand. Please try again."

# Quick manual check
print(ask_finance_assistant("What's the capital of France?"))

KeyboardInterrupt: Interrupted by user

In [ ]:
def format_summary(summary):
    lines = [
        f"Total spent: ${summary['total_spent']:.2f}",
        f"Remaining balance: ${summary['remaining_balance']:.2f}",
        f"Goal achieved: {'Yes! 🎉' if summary['goal_achieved'] else 'Not yet'}",
        "",
        "Spending by category:"
    ]
    for cat, amount in summary["category_totals"].items():
        pct = summary["category_percentages"][cat]
        lines.append(f"- {cat}: ${amount:.2f} ({pct:.1f}% of income)")
    return "\n".join(lines)


def analyse_budget(csv_file, income, goal):
    if csv_file is None:
        return "Please upload a transaction CSV file first.", None
    summary = break_down_budget(csv_file.name, income, goal)
    if summary is None:
        return "Something went wrong reading your file. Check it has Date, Amount, Category columns.", None
    return format_summary(summary), summary


def chat_with_assistant(message, budget_summary):
    if not message:
        return "Ask me anything about your budget!"
    return ask_finance_assistant(message, budget_summary=budget_summary)


with gr.Blocks(title="Budget Breaker") as demo:
    gr.Markdown("# 💸 Budget Breaker")
    gr.Markdown("Upload your transactions, set your income and savings goal, then chat with your finance assistant.")

    budget_state = gr.State(None)  # stores the last budget summary

    with gr.Row():
        csv_input = gr.File(label="Upload transactions CSV")
        income_input = gr.Number(label="Monthly income ($)", value=1500)
        goal_input = gr.Number(label="Savings goal ($)", value=300)

    analyse_btn = gr.Button("Analyse my budget")
    summary_output = gr.Textbox(label="Budget summary", lines=10)

    analyse_btn.click(
        fn=analyse_budget,
        inputs=[csv_input, income_input, goal_input],
        outputs=[summary_output, budget_state]
    )

    gr.Markdown("---")
    user_msg = gr.Textbox(label="Ask your finance assistant something")
    chat_btn = gr.Button("Ask")
    chat_output = gr.Textbox(label="Assistant's reply", lines=5)

    chat_btn.click(
        fn=chat_with_assistant,
        inputs=[user_msg, budget_state],
        outputs=chat_output
    )

demo.launch(debug=True)